In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
from pathlib import Path

In [2]:
# Custom Dataset for Multi-Raster Input
class SpatialRasterDataset(Dataset):
    """
    Dataset for loading multiple raster layers (population, GDP, land cover)
    and corresponding CISI target values
    """
    def __init__(self, pop_paths, gdp_paths, lc_paths, cisi_paths, 
                 transform=None, normalize=True):
        """
        Args:
            pop_paths: List of paths to population density rasters
            gdp_paths: List of paths to GDP rasters
            lc_paths: List of paths to land cover rasters
            cisi_paths: List of paths to CISI target rasters
            transform: Optional transform to be applied
            normalize: Whether to normalize input rasters
        """
        self.pop_paths = pop_paths
        self.gdp_paths = gdp_paths
        self.lc_paths = lc_paths
        self.cisi_paths = cisi_paths
        self.transform = transform
        self.normalize = normalize
        
    def __len__(self):
        return len(self.pop_paths)
    
    def _load_raster(self, path):
        """Load a single raster file"""
        with rasterio.open(path) as src:
            data = src.read(1)  # Read first band
            # Handle nodata values
            nodata = src.nodata
            if nodata is not None:
                data = np.where(data == nodata, 0, data)
        return data.astype(np.float32)
    
    def _normalize_data(self, data):
        """Normalize data to [0, 1] range"""
        min_val = data.min()
        max_val = data.max()
        if max_val - min_val > 0:
            return (data - min_val) / (max_val - min_val)
        return data
    
    def __getitem__(self, idx):
        # Load each raster layer
        pop = self._load_raster(self.pop_paths[idx])
        gdp = self._load_raster(self.gdp_paths[idx])
        lc = self._load_raster(self.lc_paths[idx])
        cisi = self._load_raster(self.cisi_paths[idx])
        
        # Normalize if requested
        if self.normalize:
            pop = self._normalize_data(pop)
            gdp = self._normalize_data(gdp)
            lc = self._normalize_data(lc)
            cisi = self._normalize_data(cisi)
        
        # Stack inputs into channels (3 input channels)
        # Shape: (3, H, W)
        inputs = np.stack([pop, gdp, lc], axis=0)
        
        # Target is single channel
        # Shape: (1, H, W)
        target = np.expand_dims(cisi, axis=0)
        
        # Convert to tensors
        inputs = torch.from_numpy(inputs).float()
        target = torch.from_numpy(target).float()
        
        if self.transform:
            inputs = self.transform(inputs)
            target = self.transform(target)
        
        return inputs, target

In [3]:
# Multi-Input CNN for Infrastructure Projection
class SpatialInfrastructureCNN(nn.Module):
    """
    CNN architecture for predicting CISI from multiple spatial inputs
    Designed for spatial regression with raster data
    """
    def __init__(self, input_channels=3, base_filters=64):
        super(SpatialInfrastructureCNN, self).__init__()
        
        # Encoder path with skip connections (U-Net style)
        # This preserves spatial information crucial for infrastructure mapping
        
        # Encoding blocks
        self.enc1 = self._conv_block(input_channels, base_filters)
        self.enc2 = self._conv_block(base_filters, base_filters * 2)
        self.enc3 = self._conv_block(base_filters * 2, base_filters * 4)
        self.enc4 = self._conv_block(base_filters * 4, base_filters * 8)
        
        # Bottleneck
        self.bottleneck = self._conv_block(base_filters * 8, base_filters * 16)
        
        # Decoding blocks with upsampling
        self.up4 = nn.ConvTranspose2d(base_filters * 16, base_filters * 8, 2, stride=2)
        self.dec4 = self._conv_block(base_filters * 16, base_filters * 8)
        
        self.up3 = nn.ConvTranspose2d(base_filters * 8, base_filters * 4, 2, stride=2)
        self.dec3 = self._conv_block(base_filters * 8, base_filters * 4)
        
        self.up2 = nn.ConvTranspose2d(base_filters * 4, base_filters * 2, 2, stride=2)
        self.dec2 = self._conv_block(base_filters * 4, base_filters * 2)
        
        self.up1 = nn.ConvTranspose2d(base_filters * 2, base_filters, 2, stride=2)
        self.dec1 = self._conv_block(base_filters * 2, base_filters)
        
        # Final output layer - no activation for regression
        self.out = nn.Conv2d(base_filters, 1, kernel_size=1)
        
        # Pooling
        self.pool = nn.MaxPool2d(2, 2)
        
    def _conv_block(self, in_ch, out_ch):
        """Convolutional block with batch norm and ReLU"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder with skip connections
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))
        enc4 = self.enc4(self.pool(enc3))
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool(enc4))
        
        # Decoder with skip connections
        dec4 = self.up4(bottleneck)
        dec4 = torch.cat([dec4, enc4], dim=1)
        dec4 = self.dec4(dec4)
        
        dec3 = self.up3(dec4)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.dec3(dec3)
        
        dec2 = self.up2(dec3)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.dec2(dec2)
        
        dec1 = self.up1(dec2)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.dec1(dec1)
        
        # Output - no activation for regression
        out = self.out(dec1)
        
        return out

In [4]:
# Training function with spatial metrics
def train_spatial_model(model, train_loader, val_loader, criterion, 
                        optimizer, device, epochs=50, save_path='best_model.pth'):
    """
    Train the spatial CNN model
    """
    best_val_loss = float('inf')
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        print(f'Epoch [{epoch+1}/{epochs}]')
        print(f'  Train Loss: {avg_train_loss:.6f}')
        print(f'  Val Loss: {avg_val_loss:.6f}')
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'val_loss': avg_val_loss,
            }, save_path)
            print(f'  Model saved! (Val Loss: {avg_val_loss:.6f})')
    
    return train_losses, val_losses



In [5]:
# Prediction function for SSP scenarios
def predict_future_cisi(model, pop_future, gdp_future, lc_future, device, 
                        normalize=True):
    """
    Generate future CISI predictions from SSP scenario inputs
    
    Args:
        model: Trained CNN model
        pop_future: Future population density raster (numpy array)
        gdp_future: Future GDP raster (numpy array)
        lc_future: Future land cover raster (numpy array)
        device: torch device
        normalize: Whether to normalize inputs
        
    Returns:
        Predicted CISI raster (numpy array)
    """
    model.eval()
    
    # Normalize if needed
    if normalize:
        pop_future = (pop_future - pop_future.min()) / (pop_future.max() - pop_future.min())
        gdp_future = (gdp_future - gdp_future.min()) / (gdp_future.max() - gdp_future.min())
        lc_future = (lc_future - lc_future.min()) / (lc_future.max() - lc_future.min())
    
    # Stack inputs
    inputs = np.stack([pop_future, gdp_future, lc_future], axis=0)
    inputs = torch.from_numpy(inputs).float().unsqueeze(0)  # Add batch dimension
    inputs = inputs.to(device)
    
    with torch.no_grad():
        prediction = model(inputs)
    
    # Convert back to numpy
    prediction = prediction.squeeze().cpu().numpy()
    
    return prediction


In [6]:
# Example usage
if __name__ == "__main__":
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    
    # Example: Create dummy data (replace with your actual raster paths)
    # In practice, you would load paths to your historical data
    """
    pop_paths = ['path/to/pop_1990.tif', 'path/to/pop_2000.tif', ...]
    gdp_paths = ['path/to/gdp_1990.tif', 'path/to/gdp_2000.tif', ...]
    lc_paths = ['path/to/lc_1990.tif', 'path/to/lc_2000.tif', ...]
    cisi_paths = ['path/to/cisi_1990.tif', 'path/to/cisi_2000.tif', ...]
    
    # Create datasets
    train_dataset = SpatialRasterDataset(
        pop_paths[:80], gdp_paths[:80], lc_paths[:80], cisi_paths[:80]
    )
    val_dataset = SpatialRasterDataset(
        pop_paths[80:], gdp_paths[80:], lc_paths[80:], cisi_paths[80:]
    )
    
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
    """
    
    # Initialize model
    model = SpatialInfrastructureCNN(input_channels=3, base_filters=64).to(device)
    
    # Loss function for spatial regression
    # MSE is standard, but you could also use:
    # - nn.L1Loss() for MAE
    # - Custom spatial loss that weights different regions
    criterion = nn.MSELoss()
    
    # Optimizer
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )
    
    print("\nModel Architecture:")
    print(model)
    print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Train model (uncomment when you have data)
    # train_losses, val_losses = train_spatial_model(
    #     model, train_loader, val_loader, criterion, optimizer, 
    #     device, epochs=100, save_path='cisi_model_best.pth'
    # )
    
    # For future predictions with SSP scenarios:
    """
    # Load trained model
    checkpoint = torch.load('cisi_model_best.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Load SSP scenario data for 2030, 2050, 2100
    pop_2030_ssp1 = load_raster('path/to/pop_2030_ssp1.tif')
    gdp_2030_ssp1 = load_raster('path/to/gdp_2030_ssp1.tif')
    lc_2030_ssp1 = load_raster('path/to/lc_2030_ssp1.tif')
    
    # Predict CISI for 2030 under SSP1
    cisi_2030_pred = predict_future_cisi(
        model, pop_2030_ssp1, gdp_2030_ssp1, lc_2030_ssp1, device
    )
    
    # Save prediction as raster
    # Then overlay with coastal flood inundation maps
    """
    
    print("\nModel ready for training!")
    print("\nNext steps:")
    print("1. Prepare your historical raster data (population, GDP, land cover, CISI)")
    print("2. Train the model on historical data")
    print("3. Load SSP scenario projections for 2030, 2050, 2100")
    print("4. Generate CISI predictions for each scenario")
    print("5. Overlay predictions with coastal flood maps")

Using device: cpu


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'